<a href="https://colab.research.google.com/github/eobi16/deep-learning/blob/main/Healthcareagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
# This cell is no longer needed as the OpenAI API key is now handled
# securely via Colab Secrets in cell 7u4hYHX7FSbt.
# If you still want to set NCBI_EMAIL and NCBI_TOOL directly, uncomment them below.

# NCBI_EMAIL="your_email@example.com" # Replace with your NCBI email
# NCBI_TOOL="healthcare_evidence_agent" # Replace with your NCBI tool name

In [2]:
import os
# Import the Colab userdata module to access secrets
from google.colab import userdata

# Retrieve the OpenAI API Key from Colab Secrets
# Ensure you have added OPENAI_API_KEY to Colab Secrets on the left sidebar (🔑 icon).
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# You can still use os.getenv for other variables if they are set via .env files
# or other means, but for the API key, Colab Secrets is recommended.
NCBI_EMAIL = os.getenv("NCBI_EMAIL")
NCBI_TOOL = os.getenv("NCBI_TOOL")

# Final check for the API key
if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY is missing. "
        "Please ensure it's added to Colab Secrets with the exact name 'OPENAI_API_KEY' "
        "and that 'Notebook access' is enabled for this secret."
    )

print("OpenAI API Key loaded successfully from Colab Secrets.")
# print(f"NCBI_EMAIL: {NCBI_EMAIL}") # Uncomment to verify NCBI_EMAIL if needed
# print(f"NCBI_TOOL: {NCBI_TOOL}")   # Uncomment to verify NCBI_TOOL if needed

OpenAI API Key loaded successfully from Colab Secrets.


In [3]:
print(f"The loaded OpenAI API Key (first 5 chars): {OPENAI_API_KEY[:5]}...")
print("If the key does not start with 'sk-', please re-check your Colab Secrets and re-run cell 7u4hYHX7FSbt.")

The loaded OpenAI API Key (first 5 chars): sk-pr...
If the key does not start with 'sk-', please re-check your Colab Secrets and re-run cell 7u4hYHX7FSbt.


### Steps to ensure your OpenAI API key is correctly set:

1.  **Check Colab Secrets**: Click on the '🔑' icon in the left sidebar.
2.  **Verify `OPENAI_API_KEY`**: Ensure there's a secret named `OPENAI_API_KEY` and its value is your actual OpenAI API key (starting with `sk-`), not a placeholder.
3.  **Enable Notebook Access**: Make sure 'Notebook access' is enabled for the `OPENAI_API_KEY` secret.
4.  **Re-run Cell `7u4hYHX7FSbt`**: This cell loads the key from Colab Secrets into your environment.
5.  **Re-run Cell `5K8URWNKNAAO`**: This cell initializes the OpenAI client with the loaded key.
6.  **Re-run Cell `2udWB-aCP8GY` (or your retrieval call)**: This will attempt to use the newly initialized client for embedding.

In [4]:
NCBI_EMAIL = "your_email@example.com" # IMPORTANT: Replace with your actual NCBI email
NCBI_TOOL = "healthcare_evidence_agent" # IMPORTANT: Replace with your actual NCBI tool name

print(f"NCBI_EMAIL set to: {NCBI_EMAIL}")
print(f"NCBI_TOOL set to: {NCBI_TOOL}")

NCBI_EMAIL set to: your_email@example.com
NCBI_TOOL set to: healthcare_evidence_agent


In [5]:
import requests

# NCBI_EMAIL and NCBI_TOOL are expected to be available from a previous cell (e.g., 7u4hYHX7FSbt)
# from app.config import NCBI_EMAIL, NCBI_TOOL # Removed: This import caused ModuleNotFoundError


BASE_URL = (
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
)


def search_pubmed(
    query: str,
    max_results: int = 10
):

    params = {
        "db": "pubmed",
        "term": query,
        "retmax": max_results,
        "retmode": "json",
        "email": NCBI_EMAIL,
        "tool": NCBI_TOOL,
    }

    response = requests.get(
        f"{BASE_URL}/esearch.fcgi",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    return data["esearchresult"]["idlist"]

In [6]:
# Call search_pubmed with a test query
test_query = "COVID-19 treatment"
pubmed_ids = search_pubmed(query=test_query, max_results=5)

print(f"PubMed IDs for query '{test_query}':")
if pubmed_ids:
    for pmid in pubmed_ids:
        print(pmid)
else:
    print("No results found.")

PubMed IDs for query 'COVID-19 treatment':
42609418
42609346
42609197
42609149
42608690


In [7]:
pmids = search_pubmed(
    "heart failure readmission"
)

print(pmids)

['42609336', '42609308', '42606912', '42605858', '42604414', '42604048', '42602006', '42597165', '42597114', '42596879']


In [8]:
def fetch_pubmed(
    pmids: list[str]
):

    params = {
        "db": "pubmed",
        "id": ",".join(pmids),
        "retmode": "xml",
        "email": NCBI_EMAIL,
        "tool": NCBI_TOOL,
    }

    response = requests.get(
        f"{BASE_URL}/efetch.fcgi",
        params=params,
        timeout=30
    )

    response.raise_for_status()

    return response.text

In [9]:
import xml.etree.ElementTree as ET


def parse_pubmed_xml(
    xml_text: str
):

    root = ET.fromstring(xml_text)

    papers = []

    for article in root.findall(
        ".//PubmedArticle"
    ):

        pmid = article.findtext(
            ".//PMID"
        )

        title_element = article.find(
            ".//ArticleTitle"
        )

        title = ""

        if title_element is not None:
            title = "".join(
                title_element.itertext()
            )

        abstract_parts = []

        for element in article.findall(
            ".//AbstractText"
        ):

            abstract_parts.append(
                "".join(
                    element.itertext()
                )
            )

        abstract = " ".join(
            abstract_parts
        )

        journal = article.findtext(
            ".//Journal/Title"
        )

        papers.append({
            "pmid": pmid,
            "title": title,
            "abstract": abstract,
            "journal": journal
        })

    return papers

In [10]:
pmids = search_pubmed(
    "heart failure readmission",
    max_results=20
)

xml = fetch_pubmed(pmids)

papers = parse_pubmed_xml(xml)

for paper in papers:

    print("=" * 80)

    print(
        paper["pmid"]
    )

    print(
        paper["title"]
    )

42609336
A clinical prediction model integrating cardiovascular-kidney-metabolic biomarkers for the composite outcome of quality-of-life deterioration and rehospitalization in elderly HFpEF patients.
42609308
Body roundness index as a predictor of 3-month readmission in elderly patients with first-episode acute heart failure.
42606912
Clinical Impact of Heart Failure Readmissions in HeartMate 3 Left Ventricular Assist Device Recipients.
42605858
Spontaneous coronary artery dissection versus atherosclerotic myocardial infarction in young women: a propensity-matched cohort study.
42604414
The Impact of Hospital Infrastructure Investment on Quality of Care.
42604048
Rheumatic Heart Disease in Pregnancy in Sub-Saharan Africa: A Case Series Emphasizing Gaps in Access to Definitive Care.
42602006
In-hospital inflammatory-nutritional-renal trajectory phenotypes and 90-day readmission or mortality in patients with acute decompensated heart failure: a retrospective cohort study.
42597165
Postpa

In [11]:
import json
from pathlib import Path


DATA_DIR = Path("data/papers")

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def save_papers(
    papers
):

    path = DATA_DIR / "papers.json"

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            papers,
            f,
            indent=2,
            ensure_ascii=False
        )


def load_papers():

    path = DATA_DIR / "papers.json"

    if not path.exists():

        return []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)

In [12]:
from openai import OpenAI


client = OpenAI(
    api_key=OPENAI_API_KEY
)


def embed_text(
    text: str
):
    print(f"Type of client inside embed_text: {type(client)}")
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

In [13]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found

In [14]:
import chromadb


chroma_client = chromadb.PersistentClient(
    path="data/chroma"
)

collection = chroma_client.get_or_create_collection(
    name="medical_papers"
)

In [15]:

def index_papers(
    papers
):

    for i, paper in enumerate(papers):

        text = f"""
Title:
{paper['title']}

Abstract:
{paper['abstract']}
"""

        embedding = embed_text(
            text
        )

        collection.add(
            ids=[
                paper["pmid"]
            ],
            embeddings=[
                embedding
            ],
            documents=[
                text
            ],
            metadatas=[
                {
                    "pmid": paper["pmid"],
                    "title": paper["title"],
                    "journal": paper["journal"]
                }
            ]
        )

In [16]:
def retrieve(
    query: str,
    top_k: int = 5
):

    embedding = embed_text(
        query
    )

    results = collection.query(
        query_embeddings=[
            embedding
        ],
        n_results=top_k
    )

    return results

In [18]:
results = retrieve(
    """
    What factors predict
    heart failure readmission?
    """
)

print(
    results["documents"]
)

Type of client inside embed_text: <class 'openai.OpenAI'>


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [19]:
from openai import OpenAI

# OPENAI_API_KEY is already loaded globally from cell 7u4hYHX7FSbt
# No need to import from 'app.config' here.


client = OpenAI(
    api_key=OPENAI_API_KEY
)


def generate(
    prompt: str
):

    # Note: This 'generate' function is an outdated implementation.
    # A corrected version is already defined in cell IEca4hYs4Ivt.
    # It's recommended to use the 'generate' function from cell IEca4hYs4Ivt
    # and consider removing this redundant cell (EqPKsKjJiipV).
    response = client.responses.create(
        model="gpt-5.6",
        input=prompt
    )

    return response.output_text

In [20]:
from openai import OpenAI


client = OpenAI(
    api_key=OPENAI_API_KEY
)


def generate(
    prompt: str
):

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Using a common chat model
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    return response.choices[0].message.content


In [21]:
def answer_question(
    question: str
):

    results = retrieve(
        question,
        top_k=5
    )

    documents = results[
        "documents"
    ][0]

    metadatas = results[
        "metadatas"
    ][0]

    evidence = []

    for doc, metadata in zip(
        documents,
        metadatas
    ):

        evidence.append(
            f"""
SOURCE:
PMID: {metadata['pmid']}
TITLE: {metadata['title']}

{doc}
"""
        )

    context = "\n\n".join(
        evidence
    )

    prompt = f"""
You are a healthcare research assistant.

Answer the research question using
ONLY the evidence provided.

Do not invent information.

If evidence is insufficient,
say so explicitly.

Research Question:
{question}

Evidence:
{context}

Return:

1. Summary
2. Key Findings
3. Conflicting Evidence
4. Limitations
5. Sources
"""

    return generate(
        prompt
    )

In [24]:
# main.py


question = """
What factors are associated with
hospital readmission among heart
failure patients?
"""


answer = answer_question(
    question
)

print(answer)


Type of client inside embed_text: <class 'openai.OpenAI'>


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [23]:
# The `retrieve` function is defined in cell RU2h4D8pP4uj
# The `generate` function is defined in cell IEca4hYs4Ivt
# No need to import from `app.vector_store` or `app.llm` here.


def answer_question(
    question: str
):

    results = retrieve(
        question,
        top_k=5
    )

    documents = results[
        "documents"
    ][0]

    metadatas = results[
        "metadatas"
    ][0]

    evidence = []

    for doc, metadata in zip(
        documents,
        metadatas
    ):

        evidence.append(
            f"""
SOURCE:
PMID: {metadata['pmid']}
TITLE: {metadata['title']}

{doc}
"""
        )

    context = "\n\n".join(
        evidence
    )

    prompt = f"""
You are a healthcare research assistant.

Answer the research question using
ONLY the evidence provided.

Do not invent information.

If evidence is insufficient,
say so explicitly.

Research Question:
{question}

Evidence:
{context}

Return:

1. Summary
2. Key Findings
3. Conflicting Evidence
4. Limitations
5. Sources
"""

    return generate(
        prompt
    )

In [ ]:
import json

from app.llm import generate


def create_plan(
    question: str
):

    prompt = f"""
You are a healthcare research planning agent.

Break the following question
into 3-6 independent research questions.

Original question:
{question}

Return ONLY valid JSON:

{{
    "subquestions": [
        "...",
        "...",
        "..."
    ]
}}
"""

    response = generate(
        prompt
    )

    return json.loads(
        response
    )

In [ ]:
from app.agents.planner import create_plan


plan = create_plan(
    """
    What factors are associated with
    hospital readmission among heart
    failure patients?
    """
)

print(plan)

In [ ]:
{
    "subquestions": [
        "What patient characteristics predict readmission?",
        "What clinical factors predict readmission?",
        "What medication factors predict readmission?",
        "What healthcare utilization factors predict readmission?"
    ]
}

In [ ]:
from app.retrieval import retrieve


def search_literature(
    subquestions
):

    all_results = []

    for question in subquestions:

        results = retrieve(
            question,
            top_k=5
        )

        all_results.append({
            "question": question,
            "results": results
        })

    return all_results

In [ ]:
# app/schemas.py

from pydantic import BaseModel
from typing import Optional


class StudyEvidence(
    BaseModel
):

    pmid: str

    population: str

    sample_size: Optional[int]

    study_design: str

    intervention: Optional[str]

    comparator: Optional[str]

    outcomes: list[str]

    findings: list[str]

    limitations: list[str]

In [ ]:
import json

from app.llm import generate


def extract_evidence(
    paper_text
):

    prompt = f"""
Extract structured information
from this medical research paper.

Return JSON:

{{
    "population": "",
    "sample_size": null,
    "study_design": "",
    "intervention": null,
    "comparator": null,
    "outcomes": [],
    "findings": [],
    "limitations": []
}}

Do not infer information
not contained in the paper.

Paper:

{paper_text}
"""

    result = generate(
        prompt
    )

    return json.loads(
        result
    )

In [ ]:
from app.llm import generate


def synthesize(
    question,
    evidence
):

    prompt = f"""
You are a healthcare evidence
synthesis agent.

Research question:

{question}

Structured evidence:

{evidence}

Produce:

1. Overall conclusion
2. Consistent findings
3. Conflicting findings
4. Important limitations
5. Evidence gaps

Do not claim causality
unless supported by the studies.

Clearly distinguish:
- association
- causation
- uncertainty
"""

    return generate(
        prompt
    )

In [ ]:
import json

from app.llm import generate


def verify_citation(
    claim,
    evidence
):

    prompt = f"""
You are a citation verification agent.

Determine whether the evidence
supports the claim.

Claim:
{claim}

Evidence:
{evidence}

Return JSON:

{{
    "supported": true,
    "support_level": "full",
    "explanation": ""
}}

support_level must be:

full
partial
none
"""

    result = generate(
        prompt
    )

    return json.loads(
        result
    )

In [ ]:
import json

from app.llm import generate


def quality_check(
    question,
    answer
):

    prompt = f"""
You are a quality assurance agent
for a healthcare AI system.

Review the answer for:

- hallucinations
- unsupported claims
- incorrect citations
- causal overstatement
- missing uncertainty
- contradictions
- inappropriate medical advice

Question:
{question}

Answer:
{answer}

Return JSON:

{{
    "passed": true,
    "issues": []
}}
"""

    result = generate(
        prompt
    )

    return json.loads(
        result
    )

In [ ]:
from typing import TypedDict

from langgraph.graph import (
    StateGraph,
    START,
    END
)


class ResearchState(
    TypedDict
):

    question: str

    plan: dict

    literature: list

    evidence: list

    synthesis: str

    citation_results: list

    qa: dict

In [ ]:
from app.agents.planner import create_plan


def planner_node(
    state
):

    plan = create_plan(
        state["question"]
    )

    return {
        "plan": plan
    }

In [ ]:
from app.agents.literature import (
    search_literature
)


def literature_node(
    state
):

    subquestions = state[
        "plan"
    ]["subquestions"]

    literature = search_literature(
        subquestions
    )

    return {
        "literature": literature
    }

In [ ]:
builder = StateGraph(
    ResearchState
)

builder.add_node(
    "planner",
    planner_node
)

builder.add_node(
    "literature",
    literature_node
)

builder.add_edge(
    START,
    "planner"
)

builder.add_edge(
    "planner",
    "literature"
)

builder.add_edge(
    "literature",
    END
)

graph = builder.compile()

In [ ]:
result = graph.invoke({
    "question":
        "What factors are associated with "
        "heart failure readmission?"
})

In [ ]:
def should_retry(
    state
):

    qa = state["qa"]

    if qa["passed"]:
        return "finish"

    return "research_again"

In [ ]:
builder.add_conditional_edges(
    "qa",
    should_retry,
    {
        "finish": END,
        "research_again": "literature"
    }
)

In [ ]:

import streamlit as st

from app.graph import graph


st.title(
    "Healthcare Evidence Intelligence Agent"
)

st.write(
    """
    AI system for evidence-grounded
    healthcare literature research.
    """
)


question = st.text_area(
    "Enter a research question"
)


if st.button("Research"):

    if not question:

        st.warning(
            "Enter a research question."
        )

    else:

        with st.spinner(
            "Researching..."
        ):

            result = graph.invoke({
                "question": question
            })

        st.subheader(
            "Research Plan"
        )

        st.write(
            result.get("plan")
        )

        st.subheader(
            "Evidence"
        )

        st.write(
            result.get("evidence")
        )

        st.subheader(
            "Synthesis"
        )

        st.write(
            result.get("synthesis")
        )